### Ridge Regression Implementation from Scratch

This notebook focuses on implementing **Ridge Regularization** from first principles. This implementation follows a deep dive into the linear algebra and calculus used to perform regularization in Multiple Linear Regression.

---

### 🧠 Mathematical Foundations

#### 1. Ordinary Least Squares (OLS) Loss
In standard Linear Regression, the goal is to minimize the **Residual Sum of Squares (RSS)**.

* **Summation Form:** The loss function is calculated as:
    $$\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

* **Matrix Representation:**
    If we define the error vector as $e = Y - X\beta$, then the loss is the inner product of the error vector (error transpose times error):
    $$e^Te$$

* **Expanded Matrix Form:**
    More simply, the loss function $L$ is written as:
    $$L(\beta) = (Y - X\beta)^T(Y - X\beta)$$

#### 2. The Ridge Penalty ($L_2$ Regularization)

To implement Ridge Regression, we augment the standard OLS loss function by adding an $L_2$ penalty term. This penalizes large coefficients to prevent overfitting.

**The Ridge Loss Function:**
$$L_{ridge} = (Y - X\beta)^T(Y - X\beta) + \lambda\|\beta\|^2$$

**Penalty Expansion:**
The squared magnitude of the coefficient vector, $\|\beta\|^2$, can be expressed in matrix form as the inner product of the vector with itself:
$$\lambda\|\beta\|^2 = \lambda \beta^T\beta$$

**Final Combined Objective:**
Substituting the expansion back into the loss function gives us the full objective for optimization:
$$J(\beta) = (Y - X\beta)^T(Y - X\beta) + \lambda \beta^T\beta$$
---
### 🖋️ Deriving the Ridge Regression Closed-Form Solution

To find the optimal coefficients $\beta$, we minimize the Ridge Loss Function $J(\beta)$ with respect to $\beta$.

#### 1. The Objective Function
The loss function combines the OLS loss and the $L_2$ penalty:
$$J(\beta) = (Y - X\beta)^T(Y - X\beta) + \lambda\beta^T\beta$$

#### 2. Expanding the Terms
Expanding the matrix products:
$$J(\beta) = (Y^T - \beta^TX^T)(Y - X\beta) + \lambda\beta^T\beta$$
$$J(\beta) = Y^TY - Y^TX\beta - \beta^TX^TY + \beta^TX^TX\beta + \lambda\beta^T\beta$$

Since $Y^TX\beta$ is a scalar, it is equal to its transpose $(Y^TX\beta)^T = \beta^TX^TY$. We can combine them:
$$J(\beta) = Y^TY - 2\beta^TX^TY + \beta^TX^TX\beta + \lambda\beta^T\beta$$

#### 3. Taking the Derivative
We take the partial derivative with respect to $\beta$ ($\frac{\partial J}{\partial \beta}$):
* $\frac{\partial}{\partial \beta}(Y^TY) = 0$
* $\frac{\partial}{\partial \beta}(-2\beta^TX^TY) = -2X^TY$
* $\frac{\partial}{\partial \beta}(\beta^TX^TX\beta) = 2X^TX\beta$
* $\frac{\partial}{\partial \beta}(\lambda\beta^T\beta) = 2\lambda\beta$

Setting the derivative to zero for the minimum:
$$-2X^TY + 2X^TX\beta + 2\lambda\beta = 0$$

#### 4. Solving for $\beta$
Divide the entire equation by 2 and isolate $\beta$:
$$X^TX\beta + \lambda\beta = X^TY$$

Factor out $\beta$. Note that $\lambda$ must be multiplied by the Identity Matrix $I$ to allow for matrix addition:
$$(X^TX + \lambda I)\beta = X^TY$$

Finally, multiply both sides by the inverse of $(X^TX + \lambda I)$:
$$\beta = (X^TX + \lambda I)^{-1} (X^TY)$$
For Full Derivation refer to handbook
- *Written by [Adarsh Singh]*

In [19]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
import numpy as np

In [20]:
X,Y = load_diabetes(as_frame=True,return_X_y=True)

In [21]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,Y,random_state=42,test_size=0.2)

In [22]:
from sklearn.linear_model import Ridge
R = Ridge(alpha=0.1,solver='cholesky')

R.fit(x_train,y_train)

y_pred = R.predict(x_test)

r2 = r2_score(y_test,y_pred)

print(f"r2-Score : {r2}\n")
print(f"Coefients : {R.coef_}\n")
print(f"intercept : {R.intercept_}\n")

r2-Score : 0.46085219464119265

Coefients : [  42.85566976 -205.49431899  505.08903304  317.0932049  -108.50026183
  -86.23673333 -190.36318008  151.70708637  392.28931896   79.9081772 ]

intercept : 151.45857456679613



In [23]:
class MyRidge:
    def __init__(self,alpha):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    def fit(self,x_train,y_train):
        # add [1] cols in the x_train at 0th Idx
        x_train = np.insert(x_train,0,1,axis=1)

        # Create Identity Matrix
        I = np.identity(x_train.shape[1])
        I[0][0] = 0 # sklearn wale internally intercept wale ko 0 kr dete h
        result = np.linalg.inv(np.dot(x_train.T,x_train) + self.alpha * I).dot(x_train.T).dot(y_train)

        # exracting
        self.intercept_ = result[0]
        self.coef_ = result[1:]
        
        # show 
        self.show()
    def predict(self,x_test):
        return np.dot(x_test,self.coef_) + self.intercept_
    
    def show(self):
        print(f"intercept : {self.intercept_}\n")
        print(f"coef : {self.coef_}")
        
    def evaluate(self,y_test,y_pred):
        from sklearn.metrics import r2_score,root_mean_squared_error
        print(f"r2-Score : {r2_score(y_test,y_pred)}")
        print(f"RMSE : {root_mean_squared_error(y_test,y_pred)}")

In [24]:
r1 = MyRidge(alpha=0.1)
r1.fit(x_train,y_train)
y_pred = r1.predict(x_test)

intercept : 151.45857456679607

coef : [  42.85566976 -205.49431899  505.08903304  317.0932049  -108.50026183
  -86.23673333 -190.36318008  151.70708637  392.28931896   79.9081772 ]


In [25]:
r1.evaluate(y_test,y_pred)

r2-Score : 0.46085219464119254
RMSE : 53.446111997699646
